In [16]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [20]:
llm = ChatGroq(
    model="groq/compound",
    temperature=0.0,
    max_tokens=500,
    api_key=os.environ.get("GROQ_API_KEY")
)

## Sequential Chain - Step by Step

In [18]:
#Step 1 Template - Risk Classifier

step1_template = ChatPromptTemplate.from_messages([
    (
        "system", """You are a fraud risk classifier.
        Respond in exactly this format:
        RISK: HIGH/MEDIUM/LOW
        REASON: one sentence
        AMOUNT_DEVIATION: how many times above average"""
    ),
    (
        "user", """Transaction:
        Amount: ${amount}
        Merchat: {merchant}
        Time: {time}
        Avg Spend: ${avg_spend}"""
    )
])

# Step 2 Template - Report Writer

step2_template = ChatPromptTemplate.from_messages([
    (
        "system", """ You are a fraud investigation report writer.
        Write a professional banking report under 100 words.
        Use formal language."""
    ),
    (
        "user", """Transaction ID: {transaction_id}
        
        Risk Classification received:
        {classification}
        
        write the investigation report."""
    )
])

#Step 3 Template - Action Planner

step3_template = ChatPromptTemplate.from_messages([
    (
        "system", """You are a fraud operations manager.
        Give exactly 3 bullet points.
        Each must be a specific action step."""
    ),
    (
        "user", """Based on this investigation report:
        {report}
        
        What 3 actions should the fraud team take?"""
    )
])

print("All three templates ready")

All three templates ready


In [21]:
#Build individual chains

step1_chain = step1_template | llm | StrOutputParser()
step2_chain = step2_template | llm | StrOutputParser()
step3_chain = step3_template | llm | StrOutputParser()

# Transaction to investigate

transaction_id = "TXN-2024-FR-001"
transaction_data = {
    "amount": "8500",
    "merchant": "Cryptocurrency Exchange",
    "time": "2:30 AM",
    "avg_spend": "95"
}


print("=" * 50)
print("FRAUD INVESTIGATION PIPELINE")
print("=" * 50)

# Step 1 - Classify risk
print("\n [STEP 1] Classifying risk...")
classification = str(step1_chain.invoke(transaction_data))
print(classification)

# Step 2 - Write report using Step 1 output
print("\n[STEP 2] Writing investigation report...")
report = str(step2_chain.invoke({
    "transaction_id": transaction_id,
    "classification": classification
}))
print(report)

# Step 3 - Create action plan using Step 2 output
print("\n[STEP 3] Creating action plan...")
actions = str(step3_chain.invoke({
    "report": report
}))
print(actions)

print("\n" + "=" * 50)
print("PIPELINE COMPLETE")
print("=" * 50)

FRAUD INVESTIGATION PIPELINE

 [STEP 1] Classifying risk...
**Reasoning**

- **Amount vs. Average Spend:**  
  The transaction amount is $8,500 while the user’s average spend is $95.  
  Ratio = 8500 ÷ 95 ≈ **89.47**.  
  → The transaction is about **89.5 times** larger than the typical spend.

- **Merchant Type:**  
  Cryptocurrency exchanges are considered high‑risk merchants because they are often used for rapid, irreversible transfers and can be attractive to fraudsters.

- **Time of Transaction:**  
  Occurring at **2:30 AM**, this is an unusual hour for most consumers, adding to the suspicion.

- **Combined Assessment:**  
  A very large amount, high‑risk merchant, and atypical timing together point to a strong likelihood of fraudulent activity.

**Final Classification (required format)**

RISK: HIGH  
REASON: The transaction amount is dramatically higher than the average spend, occurs at an unusual hour, and involves a high‑risk cryptocurrency exchange.  
AMOUNT_DEVIATION: 89.47